In [ ]:
import pandas as pd
import os
from functools import reduce
from io import BytesIO

import ipywidgets as widgets
from openpyxl.styles import PatternFill
from IPython.display import display, clear_output

In [ ]:
# WIDGETS ==============================================================
upload_widget = widgets.FileUpload(
    accept='.csv', multiple=False,
    description='Selecionar arquivo .csv',
    layout=widgets.Layout(width='auto')
)

use_all_toggle = widgets.ToggleButtons(
    options=['Todos os dados', 'Definir intervalo'],
    value='Todos os dados',
    style={'description_width': '0px', 'button_width': '140px'}
)

year_start_widget = widgets.BoundedIntText(
    value=1900, min=1900, max=2100,
    description='De:', style={'description_width': '30px'},
    layout=widgets.Layout(width='130px')
)
year_end_widget = widgets.BoundedIntText(
    value=2026, min=1900, max=2100,
    description='Até:', style={'description_width': '30px'},
    layout=widgets.Layout(width='130px')
)
year_range_box = widgets.HBox(
    [year_start_widget, year_end_widget],
    layout=widgets.Layout(display='none', margin='4px 0 0 0')
)

threshold_widget = widgets.BoundedIntText(
    value=25, min=1, max=31,
    description='Dias mínimos válidos por mês:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

run_button = widgets.Button(
    description=' Processar', button_style='primary', icon='play',
    layout=widgets.Layout(width='130px', height='34px')
)
download_widget = widgets.HTML(value='')
output_area     = widgets.Output()

# CALLBACKS ==============================================================
def on_toggle_change(change):
    year_range_box.layout.display = 'none' if change['new'] == 'Todos os dados' else 'flex'

use_all_toggle.observe(on_toggle_change, names='value')

def on_new_upload(change):
    if change['new']:
        output_buffer.clear()
        download_widget.value = ''
        with output_area:
            clear_output()

upload_widget.observe(on_new_upload, names='value')

# LAYOUT ================================================================
def _label(text):
    return widgets.HTML(
        f'<span style="font-weight:600;font-size:12px;color:#555;'
        f'text-transform:uppercase;letter-spacing:.5px">{text}</span>'
    )

def _divider():
    return widgets.HTML('<hr style="border:none;border-top:1px solid #e0e0e0;margin:6px 0">')

section_upload = widgets.VBox([
    _label('Arquivo'),
    upload_widget,
])

section_interval = widgets.VBox([
    _label('Período de análise'),
    use_all_toggle,
    year_range_box,
], layout=widgets.Layout(margin='0'))

section_quality = widgets.VBox([
    _label('Qualidade'),
    threshold_widget,
])

section_run = widgets.VBox([
    _label('Executar'),
    widgets.HBox([run_button, download_widget],
                 layout=widgets.Layout(align_items='center', gap='12px',)),
])

panel = widgets.VBox(
    [
        section_upload,
        _divider(),
        section_interval,
        _divider(),
        section_quality,
        _divider(),
        section_run,
        output_area,
    ],
    layout=widgets.Layout(
        padding='16px',
        border='1px solid #d0d0d0',
        border_radius='8px',
        max_width='520px',
        gap='10px',
    )
)

display(panel)

# Variável para guardar o arquivo gerado em memória
output_buffer = {}

# FUNÇÕES: CÁLCULOS COM SUFIXOS DE COLUNAS ===============================================
def calc_monthly_by_year(df_subset, suffix):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano', 'Mes'])

    # Qualidade primeiro, depois estatísticas calculadas
    result = df_subset[['EstacaoCodigo', 'Ano', 'Mes',
                         'Registros Válidos', 'Dias com Chuva',
                         'Total Mensal', 'Média Diária',
                         'Desvio Padrão Diário', 'Mediana Diária', 'Mínimo Diário',
                         'Máximo Diário']].copy()
    result = result.rename(columns={
        'Registros Válidos':    f'Registros Válidos{suffix}',
        'Dias com Chuva':       f'Dias com Chuva{suffix}',
        'Total Mensal':         f'Total Mensal{suffix}',
        'Média Diária':         f'Média Diária{suffix}',
        'Desvio Padrão Diário': f'Desvio Padrão Diário{suffix}',
        'Mediana Diária':       f'Mediana Diária{suffix}',
        'Mínimo Diário':        f'Mínimo Diário{suffix}',
        'Máximo Diário':        f'Máximo Diário{suffix}'
    })
    return result

def calc_historical_monthly(df_subset, suffix, threshold):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    # Exclui meses com registros insuficientes
    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]

    if df_valid.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Mes'])

    result = df_valid.groupby(['EstacaoCodigo', 'Mes']).agg(**{
        f'Anos Válidos{suffix}':            ('Total Mensal', 'count'),
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Dias com Chuva{suffix}':    ('Dias com Chuva', 'sum'),
        f'Média Histórica{suffix}':         ('Total Mensal', 'mean'),
        f'Desvio Padrão Histórico{suffix}': ('Total Mensal', 'std'),
        f'Mediana Histórica{suffix}':       ('Total Mensal', 'median'),
        f'Mínimo Histórico{suffix}':        ('Total Mensal', 'min'),
        f'Máximo Histórico{suffix}':        ('Total Mensal', 'max')
    }).reset_index()
    return result

def calc_historical_annual(df_subset, suffix, threshold):
    if df_subset.empty:
        return pd.DataFrame(columns=['EstacaoCodigo', 'Ano'])

    # Exclui meses com registros insuficientes antes de agregar por ano
    df_valid = df_subset[df_subset['Registros Válidos'] >= threshold]

    # Conta quantos meses válidos cada ano possui após o filtro
    valid_months_per_year = (
        df_valid.groupby(['EstacaoCodigo', 'Ano'])['Mes']
        .count()
        .reset_index()
        .rename(columns={'Mes': 'Meses Válidos'})
    )

    result = df_valid.groupby(['EstacaoCodigo', 'Ano']).agg(**{
        f'Total Registros Válidos{suffix}': ('Registros Válidos', 'sum'),
        f'Total Dias com Chuva{suffix}':    ('Dias com Chuva', 'sum'),
        f'Total Acumulado{suffix}':         ('Total Mensal', 'sum'),
        f'Média Mensal{suffix}':            ('Total Mensal', 'mean'),
        f'Desvio Padrão Mensal{suffix}':    ('Total Mensal', 'std'),
        f'Mediana Mensal{suffix}':          ('Total Mensal', 'median'),
        f'Mínimo Mensal{suffix}':           ('Total Mensal', 'min'),
        f'Máximo Mensal{suffix}':           ('Total Mensal', 'max')
    }).reset_index()

    # Adiciona coluna de meses válidos para transparência (qualidade, vem antes dos stats)
    result = result.merge(valid_months_per_year, on=['EstacaoCodigo', 'Ano'], how='left')
    result = result.rename(columns={'Meses Válidos': f'Meses Válidos{suffix}'})

    # Reposiciona Meses Válidos logo após as colunas de qualidade já inseridas
    qual_cols  = ['EstacaoCodigo', 'Ano',
                  f'Total Registros Válidos{suffix}', f'Total Dias com Chuva{suffix}',
                  f'Meses Válidos{suffix}']
    other_cols = [c for c in result.columns if c not in qual_cols]
    result = result[qual_cols + other_cols]
    return result

# LÓGICA PRINCIPAL ========================================================================
def on_run_clicked(b):
    output_buffer.clear()
    download_widget.value = ''

    with output_area:
        clear_output()

        # Verifica se arquivo foi carregado
        if not upload_widget.value:
            print("Nenhum arquivo carregado. Por favor, faça o upload de um CSV.")
            return

        if isinstance(upload_widget.value, dict):
            input_filename = list(upload_widget.value.keys())[0]
            uploaded_file  = list(upload_widget.value.values())[0]
            raw_bytes      = uploaded_file['content']
        else:
            uploaded_file  = upload_widget.value[0]
            input_filename = uploaded_file['name']
            raw_bytes      = bytes(uploaded_file['content'])

        # Lê o CSV considerando vírgulas como decimais
        df = pd.read_csv(BytesIO(raw_bytes), encoding='latin1', sep=';', skiprows=14, decimal=',')

        # EXCLUIR COLUNAS DE METADADOS
        rain_cols = [f'Chuva{i:02d}' for i in range(1, 32)]
        essential_cols = ['EstacaoCodigo', 'Data', 'NivelConsistencia']

        available_cols = [c for c in essential_cols + rain_cols if c in df.columns]
        df = df[available_cols].copy()

        date_column = 'Data'

        if date_column not in df.columns:
            print(f"AVISO: Coluna '{date_column}' não encontrada! Verifique o arquivo original.")
            return

        # Extrai Ano e Mês
        df['Data_dt'] = pd.to_datetime(df[date_column], format='%d/%m/%Y', errors='coerce')
        df['Ano'] = df['Data_dt'].dt.year
        df['Mes'] = df['Data_dt'].dt.month
        df = df.dropna(subset=['Ano', 'Mes'])

        # Converte Mês e Ano para inteiros (sem casas decimais)
        df['Ano'] = df['Ano'].astype(int)
        df['Mes'] = df['Mes'].astype(int)

        # Apaga a coluna original de data
        df = df.drop(columns=[date_column, 'Data_dt'])

        # SELEÇÃO DO INTERVALO DE ANÁLISE
        year_min = int(df['Ano'].min())
        year_max = int(df['Ano'].max())
        print(f"Dados disponíveis de {year_min} a {year_max}.")

        use_all = use_all_toggle.value == 'Todos os dados'
        if use_all:
            year_start, year_end = year_min, year_max
            print(f"Usando todos os dados ({year_min}–{year_max}, {len(df)} registros).")
        else:
            year_start = year_start_widget.value
            year_end   = year_end_widget.value

            if not (year_min <= year_start <= year_end <= year_max):
                print(f"Intervalo inválido. Use valores entre {year_min} e {year_max}, com início ≤ fim.")
                return

            df = df[(df['Ano'] >= year_start) & (df['Ano'] <= year_end)]
            print(f"Análise restrita a {year_start}–{year_end} ({len(df)} registros).")

        # LIMIAR DE QUALIDADE DE DADOS
        threshold = threshold_widget.value

        # Calcula as estatísticas diárias
        rain_cols = [c for c in df.columns if (c.startswith('Chuva') and not c.endswith('Status'))]
        df['Total Mensal']         = df[rain_cols].sum(axis=1, skipna=True)
        df['Média Diária']         = df[rain_cols].mean(axis=1, skipna=True)
        df['Desvio Padrão Diário'] = df[rain_cols].std(axis=1, skipna=True)
        df['Mediana Diária']       = df[rain_cols].median(axis=1)
        df['Mínimo Diário']        = df[rain_cols].min(axis=1)
        df['Máximo Diário']        = df[rain_cols].max(axis=1)
        df['Dias com Chuva']       = df[rain_cols].apply(lambda row: (row > 0).sum(), axis=1)
        df['Registros Válidos']    = df[rain_cols].notna().sum(axis=1)

        # SEPARAÇÃO DOS DADOS E APLICAÇÃO DOS CÁLCULOS
        df_raw        = df[df['NivelConsistencia'] == 1].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_consistent = df[df['NivelConsistencia'] == 2].drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])
        df_all        = df.sort_values(['NivelConsistencia'], ascending=False).drop_duplicates(subset=['EstacaoCodigo', 'Ano', 'Mes'])

        # TABELA 2: MENSAL POR ANO
        dfs_ma = [d for d in [
            calc_monthly_by_year(df_all,        ''),
            calc_monthly_by_year(df_raw,        ' Bruto'),
            calc_monthly_by_year(df_consistent, ' Consistido')
        ] if not d.empty]

        if dfs_ma:
            monthly_by_year = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano', 'Mes'], how='outer'), dfs_ma)
                .sort_values(['Ano', 'Mes'], ascending=[False, False])
            )
            # Reordenar: ids | qualidade agrupada por tipo | stats agrupados por tipo
            id_cols   = ['EstacaoCodigo', 'Ano', 'Mes']
            qual_names = ['Registros Válidos', 'Dias com Chuva']
            stat_names = ['Total Mensal', 'Média Diária', 'Desvio Padrão Diário',
                          'Mediana Diária', 'Mínimo Diário', 'Máximo Diário']
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in monthly_by_year.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in monthly_by_year.columns]
            monthly_by_year = monthly_by_year[id_cols + qual_cols + stat_cols]
        else:
            monthly_by_year = pd.DataFrame()

        # TABELA 3: HISTÓRICO MENSAL
        dfs_hm = [d for d in [
            calc_historical_monthly(df_all,        '',            threshold),
            calc_historical_monthly(df_raw,        ' Bruto',      threshold),
            calc_historical_monthly(df_consistent, ' Consistido', threshold)
        ] if not d.empty]

        if dfs_hm:
            monthly_stats = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Mes'], how='outer'), dfs_hm)
                .sort_values(['Mes'], ascending=True)
            )
            id_cols   = ['EstacaoCodigo', 'Mes']
            qual_names = ['Anos Válidos', 'Total Registros Válidos', 'Total Dias com Chuva']
            stat_names = ['Média Histórica', 'Desvio Padrão Histórico',
                          'Mediana Histórica', 'Mínimo Histórico', 'Máximo Histórico']
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in monthly_stats.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in monthly_stats.columns]
            monthly_stats = monthly_stats[id_cols + qual_cols + stat_cols]
        else:
            monthly_stats = pd.DataFrame()

        # TABELA 4: HISTÓRICO ANUAL
        dfs_ha = [d for d in [
            calc_historical_annual(df_all,        '',            threshold),
            calc_historical_annual(df_raw,        ' Bruto',      threshold),
            calc_historical_annual(df_consistent, ' Consistido', threshold)
        ] if not d.empty]

        if dfs_ha:
            annual_stats = (
                reduce(lambda l, r: pd.merge(l, r, on=['EstacaoCodigo', 'Ano'], how='outer'), dfs_ha)
                .sort_values(['Ano'], ascending=False)
            )
            id_cols   = ['EstacaoCodigo', 'Ano']
            qual_names = ['Total Registros Válidos', 'Total Dias com Chuva', 'Meses Válidos']
            stat_names = ['Total Acumulado', 'Média Mensal', 'Desvio Padrão Mensal',
                          'Mediana Mensal', 'Mínimo Mensal', 'Máximo Mensal']
            suffixes  = ['', ' Bruto', ' Consistido']
            qual_cols = [f'{n}{s}' for n in qual_names for s in suffixes
                         if f'{n}{s}' in annual_stats.columns]
            stat_cols = [f'{n}{s}' for n in stat_names for s in suffixes
                         if f'{n}{s}' in annual_stats.columns]
            annual_stats = annual_stats[id_cols + qual_cols + stat_cols]
        else:
            annual_stats = pd.DataFrame()

        # REORGANIZAÇÃO DA ABA PRINCIPAL (Dados Completos)

        # Ordena os dados originais do mais recente para o mais antigo
        df = df.sort_values(['Ano', 'Mes', 'NivelConsistencia'], ascending=[False, False, False])

        # Define a nova ordem exata das colunas
        base_cols       = ['EstacaoCodigo', 'Ano', 'Mes', 'NivelConsistencia']
        calculated_cols = ['Total Mensal'] # Apenas o Total foi mantido
        new_order       = [col for col in base_cols + calculated_cols + rain_cols if col in df.columns]

        # Aplica a nova ordem e remove Média e Desvio Padrão do DataFrame principal
        df = df[new_order]

        # EXPORTAR PARA EXCEL (.xlsx) EM MEMÓRIA
        time_range      = f"{year_start}_{year_end}"
        base_name, _    = os.path.splitext(input_filename)
        output_filename = f"Calculos_{base_name}_{time_range}.xlsx"

        excel_buffer = BytesIO()
        with pd.ExcelWriter(excel_buffer, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Dados_Completos', index=False)
            if not monthly_by_year.empty:
                monthly_by_year.to_excel(writer, sheet_name='Mensal_Por_Ano',   index=False)
                monthly_stats.to_excel(  writer, sheet_name='Historico_Mensal', index=False)
                annual_stats.to_excel(   writer, sheet_name='Historico_Anual',  index=False)

            # Definição das cores (códigos HEX)
            fill_header     = PatternFill(start_color="D9D9D9", end_color="D9D9D9", fill_type="solid") # Cinza claro
            fill_bruto      = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid") # Laranja claro
            fill_consistido = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid") # Azul claro

            # Itera sobre todas as abas geradas
            for sheet_name in writer.sheets:
                sheet = writer.sheets[sheet_name]
                
                # Aplica cor ao cabeçalho (linha 1)
                for cell in sheet[1]:
                    cell.fill = fill_header

                # Itera sobre as colunas para aplicar cores nos dados
                for col_idx in range(1, sheet.max_column + 1):
                    header_value = str(sheet.cell(row=1, column=col_idx).value)
                    
                    # Verifica o sufixo da coluna para determinar a cor
                    if header_value.endswith('Bruto'):
                        for row_idx in range(2, sheet.max_row + 1):
                            sheet.cell(row=row_idx, column=col_idx).fill = fill_bruto
                            
                    elif header_value.endswith('Consistido'):
                        for row_idx in range(2, sheet.max_row + 1):
                            sheet.cell(row=row_idx, column=col_idx).fill = fill_consistido

        excel_buffer.seek(0)
        excel_data = excel_buffer.read()

        print(f"Arquivo pronto: {output_filename}")
        
        # GERAÇÃO DO LINK DE DOWNLOAD HTML
        import base64
        b64 = base64.b64encode(excel_data).decode()
        mime = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
        
        download_widget.value = f'''
            <a href="data:{mime};base64,{b64}" download="{output_filename}">
                <button class="jupyter-widgets jupyter-button widget-button mod-success">
                    <i class="fa fa-download"></i> Baixar Excel
                </button>
            </a>
        '''
        download_widget.layout = widgets.Layout(width='130px', height='34px')
        
run_button.on_click(on_run_clicked)